In [1]:
import os
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import time
import scipy
from scanpy.tools._utils import get_init_pos_from_paga 
import gc

import anndata as an
import scanpy as sc

import GEOparse as geo

# import rapids_singlecell as rsc
# import scvi

# import torch
# import rmm
# import cupy
# import cudf
# import cupy as cp
# from rmm.allocators.cupy import rmm_cupy_allocator

# # Enable `managed_memory`
# rmm.reinitialize(
#     managed_memory=True,
#     pool_allocator=False,
# )
# cp.cuda.set_allocator(rmm_cupy_allocator)

"""FINAL WARNING: UserWarning silenced!"""
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logging.getLogger("GEOparse").setLevel(logging.WARNING)

In [2]:
outdir = "/scratch/indikar_root/indikar1/jrcwycy/HYB/atlas/"

gse = geo.get_GEO(geo='GSE105211', how='full', destdir=outdir)
print(gse.metadata.get('supplementary_file', None))

['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE105nnn/GSE105211/suppl/GSE105211_genes.fpkm_table.txt.gz']


In [3]:
meta = pd.DataFrame({gsm_name: gsm.metadata for gsm_name, gsm in gse.gsms.items()}).T

meta = meta.apply(lambda col: col.map(lambda x: "; ".join(x) if isinstance(x, list) else x))

print(meta.shape)
meta.head()

(658, 36)


,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,taxid_ch1,...,contact_zip/postal_code,contact_country,instrument_model,library_selection,library_source,library_strategy,relation,supplementary_file_1,series_id,data_row_count
GSM2824396,BJ_MYO_R1_T24_A01_scRNA-seq,GSM2824396,Public on Sep 26 2018,Oct 19 2017,May 15 2019,SRA,1,BJ_MYO_R1_T24,Homo sapiens,9606,...,98115,USA,Illumina HiSeq 2000,cDNA,transcriptomic,RNA-Seq,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NONE,GSE105211,0
GSM2824397,BJ_MYO_R1_T24_A02_scRNA-seq,GSM2824397,Public on Sep 26 2018,Oct 19 2017,May 15 2019,SRA,1,BJ_MYO_R1_T24,Homo sapiens,9606,...,98115,USA,Illumina HiSeq 2000,cDNA,transcriptomic,RNA-Seq,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NONE,GSE105211,0
GSM2824398,BJ_MYO_R1_T24_A03_scRNA-seq,GSM2824398,Public on Sep 26 2018,Oct 19 2017,May 15 2019,SRA,1,BJ_MYO_R1_T24,Homo sapiens,9606,...,98115,USA,Illumina HiSeq 2000,cDNA,transcriptomic,RNA-Seq,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NONE,GSE105211,0
GSM2824399,BJ_MYO_R1_T24_A04_scRNA-seq,GSM2824399,Public on Sep 26 2018,Oct 19 2017,May 15 2019,SRA,1,BJ_MYO_R1_T24,Homo sapiens,9606,...,98115,USA,Illumina HiSeq 2000,cDNA,transcriptomic,RNA-Seq,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NONE,GSE105211,0
GSM2824400,BJ_MYO_R1_T24_A05_scRNA-seq,GSM2824400,Public on Sep 26 2018,Oct 19 2017,May 15 2019,SRA,1,BJ_MYO_R1_T24,Homo sapiens,9606,...,98115,USA,Illumina HiSeq 2000,cDNA,transcriptomic,RNA-Seq,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NONE,GSE105211,0


In [5]:
cols = ['title', 'source_name_ch1', 'characteristics_ch1',
       'treatment_protocol_ch1', 'growth_protocol_ch1', 'molecule_ch1',
       'extract_protocol_ch1', 'data_processing', 'platform_id'
       ]

meta[cols].head()

,title,source_name_ch1,characteristics_ch1,treatment_protocol_ch1,growth_protocol_ch1,molecule_ch1,extract_protocol_ch1,data_processing,platform_id
GSM2824396,BJ_MYO_R1_T24_A01_scRNA-seq,BJ_MYO_R1_T24,cell type: Foreskin Human Fibroblasts; cell li...,​ ​The​ ​fibroblasts​ ​were​ ​then​ ​infected​...,​Foreskin​ ​Human​ ​Fibroblasts​ ​obtained​ ​f...,total RNA,To​ ​perform​ ​reprogramming​ ​and​ ​different...,​ ​Gene expression​ ​profiles​ ​for​ ​each​ ​c...,GPL11154
GSM2824397,BJ_MYO_R1_T24_A02_scRNA-seq,BJ_MYO_R1_T24,cell type: Foreskin Human Fibroblasts; cell li...,​ ​The​ ​fibroblasts​ ​were​ ​then​ ​infected​...,​Foreskin​ ​Human​ ​Fibroblasts​ ​obtained​ ​f...,total RNA,To​ ​perform​ ​reprogramming​ ​and​ ​different...,​ ​Gene expression​ ​profiles​ ​for​ ​each​ ​c...,GPL11154
GSM2824398,BJ_MYO_R1_T24_A03_scRNA-seq,BJ_MYO_R1_T24,cell type: Foreskin Human Fibroblasts; cell li...,​ ​The​ ​fibroblasts​ ​were​ ​then​ ​infected​...,​Foreskin​ ​Human​ ​Fibroblasts​ ​obtained​ ​f...,total RNA,To​ ​perform​ ​reprogramming​ ​and​ ​different...,​ ​Gene expression​ ​profiles​ ​for​ ​each​ ​c...,GPL11154
GSM2824399,BJ_MYO_R1_T24_A04_scRNA-seq,BJ_MYO_R1_T24,cell type: Foreskin Human Fibroblasts; cell li...,​ ​The​ ​fibroblasts​ ​were​ ​then​ ​infected​...,​Foreskin​ ​Human​ ​Fibroblasts​ ​obtained​ ​f...,total RNA,To​ ​perform​ ​reprogramming​ ​and​ ​different...,​ ​Gene expression​ ​profiles​ ​for​ ​each​ ​c...,GPL11154
GSM2824400,BJ_MYO_R1_T24_A05_scRNA-seq,BJ_MYO_R1_T24,cell type: Foreskin Human Fibroblasts; cell li...,​ ​The​ ​fibroblasts​ ​were​ ​then​ ​infected​...,​Foreskin​ ​Human​ ​Fibroblasts​ ​obtained​ ​f...,total RNA,To​ ​perform​ ​reprogramming​ ​and​ ​different...,​ ​Gene expression​ ​profiles​ ​for​ ​each​ ​c...,GPL11154


In [6]:
meta['source_name_ch1'].value_counts()

source_name_ch1
BJ_MYO_R2_T72    96
BJ_MYO_R2_T48    96
BJ_MYO_R2_T24    96
BJ_MYO_R1_T0     95
BJ_MYO_R1_T72    94
BJ_MYO_R1_T24    93
BJ_MYO_R1_T48    88
Name: count, dtype: int64